In [48]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv
/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/data_description.txt
/kaggle/input/competitions/home-data-for-ml-course/test.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv
/kaggle/input/competitions/home-data-for-ml-course/test.csv


In [49]:
from sklearn.preprocessing import OneHotEncoder
training_file_path='/kaggle/input/competitions/home-data-for-ml-course/train.csv'
training_data = pd.read_csv(training_file_path)

t = training_data.SalePrice
X = training_data.drop(['SalePrice'],axis = 1)
missing_cols = X.isna().sum()
print(missing_cols[missing_cols > 0])
redflags = ['Alley','MasVnrType','FireplaceQu','PoolQC','Fence','MiscFeature']
X = X.drop(redflags,axis=1)
X = X.drop('MiscVal',axis=1)
print(X.columns)
num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns
for col in num_cols:
    median_val = X[col].median()
    X[col] = X[col].fillna(median_val)

for col in cat_cols:
    mode_val = X[col].mode()[0]
    X[col] = X[col].fillna(mode_val)
onehotlist = ['MSSubClass','MSZoning','Street', 'LandContour', 'LotConfig', 'Neighborhood',
              'Condition1','Condition2','BldgType', 'HouseStyle', 'RoofStyle','RoofMatl',
              'Exterior1st','Exterior2nd','Foundation','Heating','CentralAir','GarageType','SaleType','SaleCondition']
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output = False)
OH_cols = pd.DataFrame(OH_encoder.fit_transform(X[onehotlist]))
OH_cols.index = X.index
num_X = X.drop(onehotlist,axis=1)
OH_X = pd.concat([num_X,OH_cols],axis=1)
OH_X.columns = OH_X.columns.astype(str)
#OH_X.head

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64
Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'Central

In [50]:
X.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
       '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
       'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF',
       'Enclo

In [54]:
from sklearn.preprocessing import OrdinalEncoder

reference = ['MSSubClass','MSZoning','Street', 'LandContour', 'LotConfig', 'Neighborhood',
              'Condition1','Condition2','BldgType', 'HouseStyle', 'RoofStyle','RoofMatl',
              'Exterior1st','Exterior2nd','Foundation','Heating','CentralAir','GarageType','SaleType','SaleCondition']

ordinalCols = ['LotShape', 'Utilities','LandSlope', 'ExterQual','ExterCond','BsmtQual',
               'BsmtCond', 'BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC',
               'Electrical','KitchenQual','Functional','GarageFinish', 'GarageQual',
               'GarageCond','PavedDrive']
lot_ranks = ["IR3","IR2","IR1","Reg"]
slope_ranks = ["Sev","Mod","Gtl"]
utility_ranks = ['ELO','NoSeWa','NoSewr','Normal','AllPub']
exposure_ranks = ['No','Mn','Av','Gd']
quality_ranks = ["Po","Fa","TA","Gd","Ex"]
paved_ranks = ["N","P","Y"]
finish_ranks = ['Unf','LwQ','Rec','BLQ','ALQ','GLQ']
electrical_ranks = ["FuseP", "FuseF", "FuseA", "Mix", "SBrkr"]
functional_ranks = ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ']
gfinish_ranks = ['NA','Unf','RFn','Fin']

categories_list= [
    lot_ranks,
    utility_ranks,
    slope_ranks,
    quality_ranks,
    quality_ranks,
    quality_ranks,
    quality_ranks,
    exposure_ranks,
    finish_ranks,
    finish_ranks,
    quality_ranks,
    electrical_ranks,
    quality_ranks,
    functional_ranks,
    gfinish_ranks,
    quality_ranks,
    quality_ranks,
    paved_ranks
]
ordinal_encoder = OrdinalEncoder(categories = categories_list)



In [55]:
final_X = OH_X.copy()
final_X[ordinalCols]= ordinal_encoder.fit_transform(OH_X[ordinalCols])
final_X = final_X.drop(["Id"],axis=1)
train_mean = np.mean(final_X, axis=0)
train_std = np.std(final_X, axis=0)
final_X = (final_X - train_mean) / train_std
final_X.shape

(1460, 218)

In [56]:
from sklearn.model_selection import train_test_split
train_X, val_X, train_y, val_y = train_test_split(final_X, t, random_state = 0)
print(train_X.shape)

(1095, 218)


In [57]:
#This is the code for which we get an initial few blackbox observations
import random
count1 = 10
first_realizations = np.random.rand(count1, 2)
first_realizations_vals = np.random.rand(count1)
fweights = np.random.rand(count1,218)
fbias = np.random.rand(count1,1)
for i in range(count1):
    first_realizations[i][0] = random.uniform(0.01,10) #learning rate
    first_realizations[i][1] = random.uniform(0.01,10) #regularization parameter
epoch_count = 3000
score = np.random.rand(epoch_count)
patience = 10
for i in range(count1):
    weights = np.zeros(218)
    bias = 0
    learning_rate= first_realizations[i][0]/100
    reg_parameter = first_realizations[i][1]*100
    #FOR THIS PART I HAVE TO IMPLEMENT EARLY STOPPING AS EPOCH COUNT IS NOT A CONTINOUS VARIABLE :(
    patience_count = 0
    mino = 10000000
    s_weights = np.zeros(218)
    s_bias = np.zeros(1)
    for j in range(epoch_count):
        y = train_X@weights+bias
        weights = weights-((learning_rate)/1095)*((train_X.T)@(y-train_y))-(1/1095)*learning_rate*reg_parameter*weights
        bias =  bias-(1/1095)*learning_rate*((y-train_y).sum())
        metric = val_X@weights+bias 
        score[j] = np.mean(np.abs(val_y-metric))
        if(score[j]>mino):
            patience_count=patience_count+1
            if(patience_count>=patience):
                break
        else:
            patience_count = 0
            mino = score[j]
            s_weights = weights
            s_bias = bias
    fweights[i,:] = s_weights
    fbias[i] = s_bias
    first_realizations_vals[i] = mino
    print(i)
print(first_realizations_vals)
current_val = 1000000
current_input = [0, 0, 0 ]
bweights = np.random.rand(218)
bbias = 0;
for i in range(count1):
    if(first_realizations_vals[i]<current_val):
        current_val = first_realizations_vals[i]
        current_input = first_realizations[i]
        bbias = fbias[i]
        bweights = fweights[i,:]

    

0
1
2
3
4
5
6
7
8
9
[20112.16675875 19877.66200648 19955.91381635 19927.10780639
 20294.78159028 19864.89791374 19865.26285531 19898.02308315
 20200.08135268 20057.15292387]


In [60]:
import math
from scipy.stats import norm
import mpmath
from mpmath import mp, matrix
from scipy.linalg import cho_factor, cho_solve
#This is the code for which we sample new points based on the argmax of acquisition function
sampled_count = 50 # number of times we update acquisition function
restart_count = 20 # number of times we perform gradient ascent for acquisition function
ascent_learning_rate = 0.005 # learning rate for gradient ascent
ascent_epoch_count = 300 # epoch count for gradient ascent
rbf_ker_paramk = 0.5
rbf_ker_paramsigma = 1
data_X = first_realizations
data_Y = first_realizations_vals
for i in range(sampled_count):
    restart_points = np.zeros((restart_count,2))
    for j in range(restart_count):
        restart_points[j][0] = random.uniform(0.01,10) #learning rate
        restart_points[j][1] = random.uniform(0.01,10) #regularization parameter
    query_point = [0,0]
    query_val = -100
    KXX = np.zeros((len(data_X),len(data_X)))
    for t in range(len(data_X)):
        for v in range(len(data_X)):
            KXX[t][v] = rbf_ker_paramk*(math.exp((-0.5/math.pow(rbf_ker_paramsigma,2))*np.linalg.norm(data_X[v]-data_X[t])*np.linalg.norm(data_X[v]-data_X[t])))
            #print(KXX[t][v])
    KXX_chol,low  = cho_factor(KXX)
    alpha = cho_solve(( KXX_chol,low),data_Y)
    KXX_inv = np.linalg.inv(KXX)
    for j in range(restart_count):
        current_vector = restart_points[j,:]
        for k in range(ascent_epoch_count):
            KxX = np.zeros(len(data_X))
            for t in range(len(KxX)):
                KxX[t] = rbf_ker_paramk*(math.exp((-0.5/math.pow(rbf_ker_paramsigma,2))*np.linalg.norm(current_vector-data_X[t])*np.linalg.norm(current_vector-data_X[t])))
            beta = cho_solve((KXX_chol,low),KxX)
            mux = KxX@alpha
            sigmasquared = rbf_ker_paramk-KxX@beta
            Z = (current_val-mux)/(math.sqrt(sigmasquared))
            cdf = norm.cdf(Z)
            pdf = norm.pdf(Z)
            jacobian = np.zeros((len(data_X),2))
            for t in range(len(data_X)):
                jacobian[t, :] = (-1/(math.pow(rbf_ker_paramsigma,2)))*rbf_ker_paramk*(math.exp((-0.5/math.pow(rbf_ker_paramsigma,2))*np.linalg.norm(current_vector-data_X[t])*np.linalg.norm(current_vector-data_X[t])))*(current_vector-data_X[t])
            gradmu = (jacobian.T)@alpha
            gradsigma= (-jacobian.T@beta)/(math.sqrt(sigmasquared))
            gradient = -gradmu*cdf+gradsigma*pdf
            current_vector+=ascent_learning_rate*gradient
            
            if(k == ascent_epoch_count-1):
                if((cdf*(current_val-mux)+(math.sqrt(sigmasquared))*pdf)>query_val):
                    query_val = (cdf*(current_val-mux)+(math.sqrt(sigmasquared))*pdf)
                    query_point = current_vector
    #At this point we should have maximized our acquisition function and have our next query point!
    nweights = np.zeros(218)
    nbias = 0
    nlearning_rate = query_point[0]/100
    nreg_parameter = query_point[1]*100
    minval = 1000000
    nscore = np.zeros(epoch_count)
    npatience_count = 0
    npatience = 10
    ns_weights = np.zeros(218)
    ns_bias = 0
    for j in range(epoch_count):
        y = train_X@nweights+nbias
        nweights = nweights-((nlearning_rate)/1095)*((train_X.T)@(y-train_y))-(1/1095)*nlearning_rate*nreg_parameter*nweights
        nbias =  nbias-(1/1095)*nlearning_rate*((y-train_y).sum())
        nmetric = val_X@nweights+nbias 
        nscore[j] = np.mean(np.abs(val_y-nmetric))
        if(nscore[j]>minval):
            npatience_count=npatience_count+1
            if(npatience_count>=npatience):
                break
        else:
            npatience_count = 0
            minval = nscore[j]
            ns_weights = nweights
            ns_bias = nbias
    print(minval)
    print(query_point)
    if(minval<current_val):
        current_val = minval
        current_input = query_point
        bweights = ns_weights
        bbias = ns_bias
    data_X = np.append(data_X, [query_point], axis = 0)
    data_Y = np.append(data_Y, [minval], axis = 0)
    print(i)
        
    
    
    

203445.5593232852
[27.89363973  1.90225376]
0
22608.572986705927
[ 1.68852302 52.23366654]
1
66074.56801362317
[  4.75225591 -11.30050216]
2
19924.747081880796
[ 0.31144792 14.5212318 ]
3
256198.7154597909
[35.45635384 -3.11591262]
4
129375.18692186744
[ 12.78057099 -43.89152053]
5
312640.6789604756
[-27.86320336  51.01327559]
6
516328.45516082115
[-54.58111944  20.17063882]
7
197882.00704138764
[ -8.11771024 -14.38669112]
8
894691.7128809144
[-102.24566031   54.15330983]
9
388222.68544483
[-37.99656169  36.04365161]
10
183805.4048582732
[ -1.36236174 -49.75920479]
11
189208.393810081
[ 25.60087872 -46.21857227]
12
386863.58546445955
[ 53.03777557 -11.62196097]
13
775412.4587022042
[102.93490893  56.15612915]
14
402000.8865709749
[55.0249054  -0.99290858]
15
221734.40341850944
[-13.84594747  17.88467746]
16
108846.66222050725
[  2.06063646 -25.01890903]
17
273812.9056991199
[-22.28314885  35.36162066]
18
194660.2287141577
[-6.99302049 58.10058891]
19
215849.07795723833
[-12.664955   -4

KeyboardInterrupt: 

In [46]:
import pandas as pd
import numpy as np
print(current_input)
# 1. Load test data
test_file_path = '/kaggle/input/competitions/home-data-for-ml-course/test.csv'
test_data = pd.read_csv(test_file_path)

# Save 'Id' column for the submission file
test_ids = test_data['Id']

# Keep only columns present in your training set (X)
test_X = test_data[X.columns].copy()
for col in num_cols:
    median_val = X[col].median()
    test_X[col] = test_X[col].fillna(median_val)

for col in cat_cols:
    mode_val = X[col].mode()[0]
    test_X[col] = test_X[col].fillna(mode_val)

OH_cols_test = pd.DataFrame(OH_encoder.transform(test_X[onehotlist]))
OH_cols_test.index = test_X.index

num_test_X = test_X.drop(onehotlist, axis=1)
OH_test_X = pd.concat([num_test_X, OH_cols_test], axis=1)
OH_test_X.columns = OH_test_X.columns.astype(str)

# 3. Apply OrdinalEncoder (.transform ONLY)
final_test_X = OH_test_X.copy()
final_test_X[ordinalCols] = ordinal_encoder.transform(OH_test_X[ordinalCols])
final_test_X = final_test_X.drop(["Id"], axis=1, errors='ignore')

final_test_X = (final_test_X - train_mean) / train_std

test_preds = final_test_X @ bweights 
test_preds +=bbias
print(test_preds)

output = pd.DataFrame({'Id': test_ids, 'SalePrice': test_preds})
output.to_csv('submission.csv', index=False)
print("Submission saved successfully!")

[0.23844883 8.403794  ]
0       109402.417198
1       167225.200204
2       184617.687330
3       195258.019047
4       205227.967100
            ...      
1454     69860.185251
1455     79514.847619
1456    171493.927499
1457    109228.822925
1458    218657.457376
Length: 1459, dtype: float64
Submission saved successfully!
